# EDA on Latest Weather Dataset

This notebook loads the latest `weather_hourly_YYYYMMDD_HHMMSS.csv` file from `data/processed/` and explores:

- time coverage and missingness
- summary tables for numeric and categorical columns
- unique values for categorical variables
- min / max ranges for numerical variables
- distributions and temporal plots for key weather signals
- hourly gaps in the raw weather feed

The dataset is already aggregated to one row per hour by `build_weather_dataset.py`.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)

In [2]:
PROJECT_DIR = Path.cwd().resolve().parent
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"


def latest_weather_file(processed_dir: Path) -> Path:
    candidates = sorted(
        path
        for path in processed_dir.glob("weather_hourly_*.csv")
        if re.fullmatch(r"weather_hourly_\d{8}_\d{6}\.csv", path.name)
    )
    if not candidates:
        raise FileNotFoundError("No timestamped weather CSV found in data/processed/.")
    return candidates[-1]

In [3]:
weather_path = latest_weather_file(PROCESSED_DIR)
weather = pd.read_csv(weather_path)

for col in ["hour_trunc", "first_timestamp", "last_timestamp"]:
    if col in weather.columns:
        weather[col] = pd.to_datetime(weather[col], errors="coerce")

print(f"Loaded weather file: {weather_path.name}")
print(f"Rows: {len(weather):,}")
print(f"Hour range: {weather['hour_trunc'].min()} -> {weather['hour_trunc'].max()}")
display(weather.head())

FileNotFoundError: No timestamped weather CSV found in data/processed/.

## 1. Dataset Snapshot

The weather table is hourly. Each row represents one hour, with summary weather variables plus metadata that indicates whether raw source observations existed in that hour.

In [ ]:
display(weather.describe(include="all").transpose())

## 2. Missingness Overview

This table shows which columns contain missing values in the hourly exploration dataset.

In [ ]:
missing_summary = (
    pd.DataFrame({
        "missing_count": weather.isna().sum(),
        "missing_pct": 100 * weather.isna().mean(),
    })
    .sort_values("missing_count", ascending=False)
)
display(missing_summary)

plt.figure(figsize=(10, 6))
missing_summary.query("missing_count > 0").sort_values("missing_count").plot(
    kind="barh", y="missing_count", legend=False, color="#C44E52"
)
plt.title("Missing Values by Column")
plt.xlabel("Missing count")
plt.ylabel("")
plt.tight_layout()
plt.show()

## 3. Categorical Columns: Unique Values

For weather exploration, the most relevant categorical-style columns are usually boolean / code columns rather than free-text categories.

This section lists unique values and their frequencies.

In [ ]:
categorical_cols = ["has_raw_observation", "weathercode"]

for col in categorical_cols:
    print(f"\nUnique values for {col}:")
    display(weather[col].value_counts(dropna=False).rename_axis(col).reset_index(name="count"))

## 4. Numerical Columns: Min / Max / Range

This section provides a compact table for the numeric weather variables, including min and max values.

In [ ]:
numeric_cols = [
    "raw_observation_count",
    "temperature",
    "precipitation",
    "windspeed",
    "cloudcover",
]

numeric_summary = pd.DataFrame({
    "min": weather[numeric_cols].min(),
    "max": weather[numeric_cols].max(),
    "mean": weather[numeric_cols].mean(),
    "std": weather[numeric_cols].std(),
    "missing_count": weather[numeric_cols].isna().sum(),
}).sort_index()

display(numeric_summary)

## 5. Time Coverage and Gaps

The key diagnostic here is whether the source weather feed populated every hour. `has_raw_observation = False` means that hour exists in the hourly table only because we expanded to a complete hourly index.

In [ ]:
coverage_summary = pd.DataFrame({
    "hours_with_raw_observation": [int(weather['has_raw_observation'].sum())],
    "hours_without_raw_observation": [int((~weather['has_raw_observation']).sum())],
    "pct_hours_without_raw_observation": [100 * (~weather['has_raw_observation']).mean()],
})
display(coverage_summary)

gap_rows = weather.loc[~weather['has_raw_observation'], ['hour_trunc']].copy()
print("First missing weather hours:")
display(gap_rows.head(30))

In [ ]:
daily_coverage = (
    weather.assign(service_date=weather['hour_trunc'].dt.date)
    .groupby('service_date')
    .agg(
        total_hours=('hour_trunc', 'size'),
        observed_hours=('has_raw_observation', 'sum'),
    )
    .reset_index()
)
daily_coverage['missing_hours'] = daily_coverage['total_hours'] - daily_coverage['observed_hours']

display(daily_coverage.sort_values('missing_hours', ascending=False).head(15))

plt.figure(figsize=(12, 5))
sns.lineplot(data=daily_coverage, x='service_date', y='missing_hours', marker='o', color='#C44E52')
plt.title('Missing Raw Weather Hours per Day')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. Distributions of Main Weather Variables

These histograms show the distribution of the main weather measurements across the hourly table.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

sns.histplot(weather['temperature'].dropna(), bins=30, ax=axes[0, 0], color='#DD8452')
axes[0, 0].set_title('Temperature Distribution')

sns.histplot(weather['precipitation'].dropna(), bins=30, ax=axes[0, 1], color='#4C72B0')
axes[0, 1].set_title('Precipitation Distribution')

sns.histplot(weather['windspeed'].dropna(), bins=30, ax=axes[1, 0], color='#55A868')
axes[1, 0].set_title('Windspeed Distribution')

sns.histplot(weather['cloudcover'].dropna(), bins=30, ax=axes[1, 1], color='#8172B3')
axes[1, 1].set_title('Cloudcover Distribution')

plt.tight_layout()
plt.show()

## 7. Weather Through Time

These plots show how the hourly variables evolve over time.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True)

axes[0].plot(weather['hour_trunc'], weather['temperature'], color='#DD8452')
axes[0].set_title('Temperature Over Time')

axes[1].plot(weather['hour_trunc'], weather['precipitation'], color='#4C72B0')
axes[1].set_title('Precipitation Over Time')

axes[2].plot(weather['hour_trunc'], weather['windspeed'], color='#55A868')
axes[2].set_title('Windspeed Over Time')

axes[3].plot(weather['hour_trunc'], weather['cloudcover'], color='#8172B3')
axes[3].set_title('Cloudcover Over Time')

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 8. Hour-of-Day Patterns

A simple way to inspect weather seasonality is to average each variable by hour of day.

In [ ]:
hourly_pattern = (
    weather.assign(hour=weather['hour_trunc'].dt.hour)
    .groupby('hour')
    .agg(
        temperature=('temperature', 'mean'),
        precipitation=('precipitation', 'mean'),
        windspeed=('windspeed', 'mean'),
        cloudcover=('cloudcover', 'mean'),
    )
    .reset_index()
)

fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=True)

sns.lineplot(data=hourly_pattern, x='hour', y='temperature', marker='o', ax=axes[0, 0], color='#DD8452')
axes[0, 0].set_title('Average Temperature by Hour')

sns.lineplot(data=hourly_pattern, x='hour', y='precipitation', marker='o', ax=axes[0, 1], color='#4C72B0')
axes[0, 1].set_title('Average Precipitation by Hour')

sns.lineplot(data=hourly_pattern, x='hour', y='windspeed', marker='o', ax=axes[1, 0], color='#55A868')
axes[1, 0].set_title('Average Windspeed by Hour')

sns.lineplot(data=hourly_pattern, x='hour', y='cloudcover', marker='o', ax=axes[1, 1], color='#8172B3')
axes[1, 1].set_title('Average Cloudcover by Hour')

plt.tight_layout()
plt.show()

## 9. Weathercode Summary

Weather codes are useful as discrete condition labels. This table lists their observed frequencies.

In [ ]:
weathercode_summary = (
    weather['weathercode']
    .value_counts(dropna=False)
    .rename_axis('weathercode')
    .reset_index(name='count')
)
display(weathercode_summary)

plt.figure(figsize=(10, 5))
sns.barplot(data=weathercode_summary, x='weathercode', y='count', color='#4C72B0')
plt.title('Weathercode Frequency')
plt.tight_layout()
plt.show()

## 10. Interpretation Notes

- `has_raw_observation = False` identifies hours that were added when expanding the table to a complete hourly index.
- If many modeling rows later miss weather, this table is the right place to verify whether the issue is caused by gaps in the raw source feed.
- `weathercode` should be treated as a categorical signal in downstream models, while `temperature`, `precipitation`, `windspeed`, and `cloudcover` are numerical features.
- The min/max table is the quickest sanity check for unrealistic values or unit problems.